# Hindu Temple Classification — CLIP Linear Probe

Freeze CLIP, embed images, train a linear classifier on top. Compares against zero-shot CLIP as a baseline.

Runtime: GPU (T4 is fine).

In [ ]:
!pip install git+https://github.com/openai/CLIP.git ftfy regex kaggle scikit-learn -q

## 1. Get the dataset
Upload your `kaggle.json` API token when prompted (Kaggle account → Settings → Create New Token).

In [ ]:
from google.colab import files
files.upload()  # select kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d tarundalal/indian-temples-dataset -p data --unzip

## 2. Load CLIP and the dataset
`ImageFolder` treats each subfolder under the dataset root as a class — no manual labeling needed.

In [ ]:
import os, torch, clip
from torchvision.datasets import ImageFolder

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

def find_dataset_root(base="data"):
    # Kaggle zips sometimes nest an extra folder; walk down to the first
    # level that actually has multiple class subfolders.
    for dirpath, dirnames, _ in os.walk(base):
        if len(dirnames) > 1:
            return dirpath
    return base

root = find_dataset_root()
print("Dataset root:", root)
dataset = ImageFolder(root, transform=preprocess)
classes = dataset.classes
print(f"{len(classes)} classes, {len(dataset)} images")
print(classes)

In [ ]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(
    dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)
print(f"train={len(train_ds)} val={len(val_ds)}")

## 3. Embed images (frozen CLIP, no training)

In [ ]:
import numpy as np
from torch.utils.data import DataLoader

@torch.no_grad()
def get_embeddings(ds, batch_size=64):
    loader = DataLoader(ds, batch_size=batch_size, num_workers=2)
    feats, labels = [], []
    for images, targets in loader:
        images = images.to(device)
        emb = model.encode_image(images)
        emb /= emb.norm(dim=-1, keepdim=True)
        feats.append(emb.cpu().numpy())
        labels.append(targets.numpy())
    return np.concatenate(feats), np.concatenate(labels)

train_x, train_y = get_embeddings(train_ds)
val_x, val_y = get_embeddings(val_ds)

## 4. Train the linear probe
This *is* the training step — a logistic regression on frozen CLIP embeddings. Seconds, not epochs.

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000, C=0.316)
clf.fit(train_x, train_y)
probe_acc = clf.score(val_x, val_y)
print(f"Linear probe val accuracy: {probe_acc:.3f}")

## 5. Zero-shot baseline (proves the probe was worth it)

In [ ]:
prompts = [f"a photo of the {c.replace('_', ' ')}, an Indian Hindu temple" for c in classes]
text_tokens = clip.tokenize(prompts).to(device)

with torch.no_grad():
    text_features = model.encode_text(text_tokens)
    text_features /= text_features.norm(dim=-1, keepdim=True)

val_image_features = torch.from_numpy(val_x).to(device)
similarity = val_image_features @ text_features.T
zero_shot_preds = similarity.argmax(dim=-1).cpu().numpy()
zero_shot_acc = (zero_shot_preds == val_y).mean()
print(f"Zero-shot CLIP val accuracy: {zero_shot_acc:.3f}")

print(f"\nLinear probe: {probe_acc:.3f}  |  Zero-shot: {zero_shot_acc:.3f}  |  Gain: {probe_acc - zero_shot_acc:+.3f}")

## 6. Save the probe

In [ ]:
import pickle
from google.colab import drive

drive.mount('/content/drive')
with open('/content/drive/MyDrive/temple_clip_probe.pkl', 'wb') as f:
    pickle.dump({'clf': clf, 'classes': classes}, f)
print("Saved to /content/drive/MyDrive/temple_clip_probe.pkl")

## 7. Inference on a new image

In [ ]:
from PIL import Image

@torch.no_grad()
def predict(image_path):
    img = preprocess(Image.open(image_path)).unsqueeze(0).to(device)
    emb = model.encode_image(img)
    emb /= emb.norm(dim=-1, keepdim=True)
    pred = clf.predict(emb.cpu().numpy())[0]
    return classes[pred]

# predict("/path/to/some_temple.jpg")